# 02 - Corn-weighted MODIS NDVI/EVI: exploratory checks

Reads `data/processed/modis_state.parquet` (built by `scripts/build_modis.py`). One row per state, platform (Terra/Aqua) and 16-day composite. Values are corn-fraction weighted (CDL years before the season only). `available_date = end_date + LAG_DAYS` is the first date a row may be used in the backtest.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from cornsat.modis import LAG_DAYS, load_modis_state

df = load_modis_state()
print(df.shape, 'LAG_DAYS =', LAG_DAYS)
df.groupby(['year', 'state']).size().unstack()

## Seasonal curves by state (each line a year; 2012 drought in red)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)
for ax, (st, g) in zip(axes, df.groupby('state')):
    for yr, gy in g.groupby('year'):
        doy = gy['mid_date'].dt.dayofyear
        kw = dict(color='red', lw=2.5, label='2012') if yr == 2012 else dict(color='grey', lw=0.7, alpha=0.6)
        ax.plot(doy, gy['ndvi_mean'], **kw)
    ax.set_title(st); ax.set_xlabel('day of year (mid composite)')
axes[0].set_ylabel('corn-weighted NDVI'); axes[0].legend()
plt.tight_layout()

## July and July-August means, peak timing

In [ ]:
jul = df[df['mid_date'].dt.month == 7]
jul.groupby(['year', 'state'])[['ndvi_mean', 'evi_mean']].mean().round(3).unstack('state')

In [ ]:
ja = df[df['mid_date'].dt.month.isin([7, 8])]
ja.groupby(['year', 'state'])['evi_mean'].mean().round(3).unstack('state').plot(marker='o', figsize=(10, 4), title='Jul-Aug mean corn EVI')

In [ ]:
def peak(g):
    g = g.sort_values('mid_date')
    r = g['ndvi_mean'].rolling(3, center=True).mean()
    return g.loc[r.idxmax(), 'mid_date'].date() if r.notna().any() else None

df.groupby(['year', 'state']).apply(peak, include_groups=False).unstack('state')

## Terra vs Aqua and data quality

In [ ]:
s = df[df['state'] == 'IL'].set_index('mid_date').sort_index()
ax = s[s.platform == 'terra']['ndvi_mean'].plot(style='o-', ms=3, label='terra', figsize=(14, 3))
s[s.platform == 'aqua']['ndvi_mean'].plot(style='s-', ms=3, ax=ax, label='aqua'); ax.legend(); ax.set_title('IL corn NDVI by platform')

In [ ]:
print('mean valid_frac', df['valid_frac'].mean().round(3), '| good-only', df['valid_frac_good'].mean().round(3))
df.groupby('year')[['valid_frac', 'valid_frac_good']].mean().round(3)